# 🏠 House Price Prediction — Pipeline MLOps complet sur Snowflake

## Contexte
Ce notebook implémente un pipeline **Data Engineering & Machine Learning** de bout en bout sur Snowflake pour prédire le **prix de vente de maisons** à partir de leurs caractéristiques.

### Étapes couvertes :
1. **Configuration de l'environnement** — Session, base de données, schéma, stage
2. **Ingestion des données** — Chargement depuis S3 via un stage externe
3. **Exploration des données (EDA)** — Statistiques descriptives, corrélations
4. **Préparation des features** — Encodage, normalisation, train/test split
5. **Entraînement des modèles** — Linear Regression, Random Forest, XGBoost
6. **Évaluation** — RMSE, MAE, R²
7. **Optimisation** — GridSearchCV sur XGBoost
8. **Model Registry** — Stockage du meilleur modèle avec métriques
9. **Inférence** — Prédictions depuis le registry
10. **Application Streamlit** — Interface utilisateur interactive

**Dataset** : `s3://logbrain-datalake/datasets/house_price/`

---
## ⚙️ Étape 1 — Configuration de l'environnement

Nous commençons par créer la base de données, le schéma, le file format CSV et le stage externe pointant vers le bucket S3.

In [ ]:
-- Création de l'environnement de travail
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;
USE DATABASE HOUSE_PRICE_DB;

CREATE SCHEMA IF NOT EXISTS ML_SCHEMA;
USE SCHEMA ML_SCHEMA;

-- File format CSV
CREATE FILE FORMAT IF NOT EXISTS CSV_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('');

-- Stage externe pointant vers S3
CREATE STAGE IF NOT EXISTS HOUSE_PRICE_STAGE
    FILE_FORMAT = CSV_FORMAT
    URL = 's3://logbrain-datalake/datasets/house_price/';

-- Stage interne pour stocker les artefacts ML (pipeline, modèles)
CREATE STAGE IF NOT EXISTS ML_ASSETS;

In [ ]:
-- Vérifier les fichiers disponibles dans le stage
LS @HOUSE_PRICE_STAGE;

In [ ]:
# Initialisation de la session Snowpark
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql_simplifier_enabled = True

# Query tag pour le monitoring et le troubleshooting
session.query_tag = {
    "origin": "MBAESG",
    "name": "house_price_prediction",
    "version": {"major": 1, "minor": 0},
    "attributes": {"is_evaluation": 1, "source": "notebook"}
}

print('✅ Session Snowpark établie avec succès !')
print(f'User      : {session.get_current_user()}')
print(f'Role      : {session.get_current_role()}')
print(f'Database  : {session.get_current_database()}')
print(f'Schema    : {session.get_current_schema()}')
print(f'Warehouse : {session.get_current_warehouse()}')

In [ ]:
# Vérification des dépendances Python disponibles
import sklearn; print(f'scikit-learn : {sklearn.__version__}')
import xgboost; print(f'xgboost      : {xgboost.__version__}')
import pandas; print(f'pandas       : {pandas.__version__}')
import numpy; print(f'numpy        : {numpy.__version__}')
import matplotlib; print(f'matplotlib   : {matplotlib.__version__}')
import seaborn; print(f'seaborn      : {seaborn.__version__}')
import snowflake.ml; print(f'snowflake-ml : OK')
print('\n✅ Toutes les dépendances sont disponibles.')

---
## 📥 Étape 2 — Ingestion des données depuis S3

Nous utilisons **Snowpark DataFrameReader** pour charger le fichier CSV depuis le stage externe.
L'option `infer_schema=True` permet à Snowflake de déduire automatiquement les types de colonnes.

In [ ]:
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import IntegerType, FloatType, StringType

# ── Détection automatique du format de fichier dans le stage ──────────────
# Le stage peut contenir des fichiers CSV ou JSON.
# On détecte le format depuis l'extension du fichier et on adapte le chargement.

# 1. Recréer un stage propre sans file format attaché
session.sql("""
    CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
    URL = 's3://logbrain-datalake/datasets/house_price/'
""").collect()

# 2. Lister les fichiers dans le stage et détecter le format
stage_files = session.sql('LS @HOUSE_PRICE_STAGE').collect()
print('📁 Fichiers détectés dans le stage :')
for f in stage_files:
    print(f'   {f["name"]}')

# Détection du format depuis l'extension du premier fichier
first_file = stage_files[0]['name'].lower() if stage_files else ''
if '.json' in first_file:
    FILE_FORMAT = 'JSON'
elif '.csv' in first_file or '.gz' in first_file:
    FILE_FORMAT = 'CSV'
else:
    FILE_FORMAT = 'CSV'  # fallback

print(f'\n🔍 Format détecté : {FILE_FORMAT}')

# ── DDL de la table cible (commun CSV et JSON) ────────────────────────────
session.sql("""
CREATE OR REPLACE TABLE HOUSE_PRICE_RAW (
    PRICE             FLOAT,
    AREA              INTEGER,
    BEDROOMS          INTEGER,
    BATHROOMS         INTEGER,
    STORIES           INTEGER,
    MAINROAD          STRING,
    GUESTROOM         STRING,
    BASEMENT          STRING,
    HOTWATERHEATING   STRING,
    AIRCONDITIONING   STRING,
    PARKING           INTEGER,
    PREFAREA          STRING,
    FURNISHINGSTATUS  STRING
)""").collect()

# ── Chargement CSV ────────────────────────────────────────────────────────
if FILE_FORMAT == 'CSV':
    copy_result = session.sql("""
        COPY INTO HOUSE_PRICE_RAW
        FROM @HOUSE_PRICE_STAGE
        FILE_FORMAT = (
            TYPE            = 'CSV'
            FIELD_DELIMITER = ','
            SKIP_HEADER     = 1
            FIELD_OPTIONALLY_ENCLOSED_BY = '"'
            NULL_IF          = ('')
            EMPTY_FIELD_AS_NULL = TRUE
        )
        ON_ERROR = 'CONTINUE'
    """).collect()

    print('\n✅ Chargement CSV — résultat COPY INTO :')
    for row in copy_result:
        print(f'   Fichier: {row[0]} | Statut: {row[1]} | Lignes chargées: {row[3]}')

# ── Chargement JSON ───────────────────────────────────────────────────────
elif FILE_FORMAT == 'JSON':
    # Étape 1 : charger le JSON brut dans une table de staging
    session.sql("""
        CREATE OR REPLACE TABLE HOUSE_PRICE_JSON_RAW (V VARIANT)
    """).collect()

    session.sql("""
        COPY INTO HOUSE_PRICE_JSON_RAW
        FROM @HOUSE_PRICE_STAGE
        FILE_FORMAT = (
            TYPE             = 'JSON'
            STRIP_OUTER_ARRAY = TRUE
        )
        ON_ERROR = 'CONTINUE'
    """).collect()

    # Étape 2 : parser le JSON et insérer dans la table structurée
    session.sql("""
        INSERT INTO HOUSE_PRICE_RAW
        SELECT
            V:price::FLOAT            AS PRICE,
            V:area::INTEGER           AS AREA,
            V:bedrooms::INTEGER       AS BEDROOMS,
            V:bathrooms::INTEGER      AS BATHROOMS,
            V:stories::INTEGER        AS STORIES,
            V:mainroad::STRING        AS MAINROAD,
            V:guestroom::STRING       AS GUESTROOM,
            V:basement::STRING        AS BASEMENT,
            V:hotwaterheating::STRING AS HOTWATERHEATING,
            V:airconditioning::STRING AS AIRCONDITIONING,
            V:parking::INTEGER        AS PARKING,
            V:prefarea::STRING        AS PREFAREA,
            V:furnishingstatus::STRING AS FURNISHINGSTATUS
        FROM HOUSE_PRICE_JSON_RAW
    """).collect()

    print('\n✅ Chargement JSON — parsing et insertion terminés.')

# ── Lecture finale via Snowpark ───────────────────────────────────────────
df_raw = session.table('HOUSE_PRICE_RAW')
print(f'\n✅ Table HOUSE_PRICE_RAW prête : {df_raw.count()} lignes, {len(df_raw.columns)} colonnes')
df_raw

In [ ]:
# La table HOUSE_PRICE_RAW est déjà créée par COPY INTO ci-dessus.
# On recharge via session.table() pour avoir le DataFrame Snowpark prêt.
df_raw = session.table('HOUSE_PRICE_RAW')

# Vérification rapide
print(f'Colonnes : {df_raw.columns}')
print(f'Lignes   : {df_raw.count()}')
df_raw.describe()

In [ ]:
-- Aperçu de la table brute
SELECT * FROM HOUSE_PRICE_RAW LIMIT 10;

---
## 🔍 Étape 3 — Exploration des données (EDA)

Avant de construire un modèle, il est essentiel de bien comprendre les données :
- Statistiques descriptives
- Distribution de la variable cible (`price`)
- Corrélations entre les variables numériques
- Distribution des variables catégorielles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter('ignore')

# Chargement depuis la table Snowflake
df = session.table('HOUSE_PRICE_RAW')

# Conversion en Pandas pour l'EDA visuelle
df_pd = df.to_pandas()

print(f'Shape : {df_pd.shape}')
print(f'\nTypes de colonnes :')
print(df_pd.dtypes)

In [ ]:
# Statistiques descriptives
print('=== Statistiques descriptives (colonnes numériques) ===')
df_pd.describe().round(2)

In [ ]:
# Vérification des valeurs manquantes
missing = df_pd.isnull().sum()
print('=== Valeurs manquantes par colonne ===')
print(missing[missing > 0] if missing.sum() > 0 else '✅ Aucune valeur manquante détectée.')

In [ ]:
# Distribution de la variable cible : PRICE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(df_pd['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du prix (PRICE)', fontsize=13)
axes[0].set_xlabel('Prix')
axes[0].set_ylabel('Fréquence')

# Boxplot
axes[1].boxplot(df_pd['PRICE'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Boxplot du prix (PRICE)', fontsize=13)
axes[1].set_xlabel('Prix')

plt.tight_layout()
plt.show()

print(f"Prix min    : {df_pd['PRICE'].min():,.0f}")
print(f"Prix max    : {df_pd['PRICE'].max():,.0f}")
print(f"Prix médian : {df_pd['PRICE'].median():,.0f}")
print(f"Prix moyen  : {df_pd['PRICE'].mean():,.0f}")

In [ ]:
# Matrice de corrélation — variables numériques
# On encode les colonnes binaires yes/no en 0/1 pour inclure leur corrélation
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']
df_corr = df_pd.copy()
for col in binary_cols:
    df_corr[col] = df_corr[col].map({'yes': 1, 'no': 0})

# Encodage ordinal pour furnishingstatus
df_corr['FURNISHINGSTATUS'] = df_corr['FURNISHINGSTATUS'].map(
    {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
)

corr_matrix = df_corr.corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Matrice de corrélation — House Price Dataset', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Top corrélations avec PRICE
print('\n=== Top corrélations avec PRICE ===')
print(corr_matrix['PRICE'].sort_values(ascending=False).drop('PRICE').round(3))

In [ ]:
# Distribution des variables catégorielles & impact sur le prix
cat_cols = ['FURNISHINGSTATUS', 'MAINROAD', 'AIRCONDITIONING', 'PREFAREA']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df_pd.groupby(col)['PRICE'].median().sort_values().plot(
        kind='bar', ax=axes[i], color='steelblue', edgecolor='white'
    )
    axes[i].set_title(f'Prix médian par {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Prix médian')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Impact des variables catégorielles sur le prix', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots : variables numériques vs PRICE
num_cols = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']

fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 4))
for i, col in enumerate(num_cols):
    axes[i].scatter(df_pd[col], df_pd['PRICE'], alpha=0.5, color='steelblue', s=20)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('PRICE' if i == 0 else '')
    axes[i].set_title(f'{col} vs PRICE')

plt.suptitle('Relations entre variables numériques et PRICE', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 🛠️ Étape 4 — Préparation des features (Feature Engineering)

Nous allons :
1. **Encoder** les variables binaires (yes/no → 0/1)
2. **Encoder ordinalement** `furnishingstatus` (furnished > semi-furnished > unfurnished)
3. **Normaliser** les variables numériques continues avec `MinMaxScaler`
4. **Séparer** X (features) et y (target)
5. **Splitter** en train/test (80/20)
6. **Sauvegarder** les datasets dans Snowflake

In [ ]:
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import IntegerType, FloatType

# Rechargement de la table brute via Snowpark
df_sp = session.table('HOUSE_PRICE_RAW')

# --- Encodage des variables binaires yes/no → 1/0 ---
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
               'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']

for col in binary_cols:
    df_sp = df_sp.with_column(
        col,
        F.when(F.col(col) == F.lit('yes'), F.lit(1)).otherwise(F.lit(0)).cast(IntegerType())
    )

# --- Encodage ordinal de furnishingstatus ---
df_sp = df_sp.with_column(
    'FURNISHINGSTATUS',
    F.when(F.col('FURNISHINGSTATUS') == F.lit('furnished'),      F.lit(2))
     .when(F.col('FURNISHINGSTATUS') == F.lit('semi-furnished'), F.lit(1))
     .otherwise(F.lit(0))
     .cast(IntegerType())
)

print('✅ Encodage des variables catégorielles terminé.')
df_sp

In [ ]:
import snowflake.ml.modeling.preprocessing as snowml
from snowflake.ml.modeling.pipeline import Pipeline
import pickle

# Colonnes à normaliser (variables numériques continues)
NUMERICAL_COLS = ['AREA', 'PRICE']
FEATURE_COLS   = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES',
                  'MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING',
                  'AIRCONDITIONING', 'PARKING', 'PREFAREA', 'FURNISHINGSTATUS']
LABEL_COL      = 'PRICE'

# Pipeline de prétraitement : MinMaxScaler sur AREA
preprocessing_pipeline = Pipeline(steps=[
    ('MMS', snowml.MinMaxScaler(
        clip=True,
        input_cols=['AREA'],
        output_cols=['AREA']
    ))
])

# Fit + Transform
df_prepared = preprocessing_pipeline.fit(df_sp).transform(df_sp)

# Sauvegarde locale du pipeline pour le registry
PIPELINE_FILE = '/tmp/preprocessing_pipeline.pkl'
with open(PIPELINE_FILE, 'wb') as f:
return pickle.load(f)
pickle.dump(preprocessing_pipeline, f)
session.file.put(PIPELINE_FILE, '@ML_ASSETS', overwrite=True)

print('✅ Normalisation appliquée et pipeline sauvegardé dans @ML_ASSETS.')
df_prepared

In [ ]:
# Split train / test avec Snowpark (80/20, seed=42 pour reproductibilité)
df_train_sp, df_test_sp = df_prepared.random_split(weights=[0.8, 0.2], seed=42)

# Sauvegarde des datasets en tables Snowflake
df_train_sp.write.mode('overwrite').save_as_table('HOUSE_PRICE_TRAIN')
df_test_sp.write.mode('overwrite').save_as_table('HOUSE_PRICE_TEST')

train_count = df_train_sp.count()
test_count  = df_test_sp.count()
total       = train_count + test_count

print(f'✅ Split terminé :')
print(f'   Train : {train_count} lignes ({train_count/total*100:.1f}%)')
print(f'   Test  : {test_count} lignes ({test_count/total*100:.1f}%)')

In [ ]:
# Conversion en Pandas pour scikit-learn / XGBoost
train_pd = df_train_sp.select(FEATURE_COLS + [LABEL_COL]).to_pandas()
test_pd  = df_test_sp.select(FEATURE_COLS + [LABEL_COL]).to_pandas()

X_train = train_pd[FEATURE_COLS]
y_train = train_pd[LABEL_COL]
X_test  = test_pd[FEATURE_COLS]
y_test  = test_pd[LABEL_COL]

print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')

---
## 🤖 Étape 5 — Entraînement des modèles ML

Nous allons entraîner **3 modèles** de régression :
1. **Linear Regression** (baseline)
2. **Random Forest Regressor**
3. **XGBoost Regressor**

Les métriques utilisées sont celles de la **régression** : RMSE, MAE, R².

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model(name, y_true, y_pred_train, y_pred_test, y_train_true):
    """Calcule et affiche les métriques RMSE, MAE, R² pour train et test."""
    rmse_train = np.sqrt(mean_squared_error(y_train_true, y_pred_train))
    rmse_test  = np.sqrt(mean_squared_error(y_true, y_pred_test))
    mae_train  = mean_absolute_error(y_train_true, y_pred_train)
    mae_test   = mean_absolute_error(y_true, y_pred_test)
    r2_train   = r2_score(y_train_true, y_pred_train)
    r2_test    = r2_score(y_true, y_pred_test)
    overfit    = r2_train - r2_test

    print(f'\n📊 [{name}]')
    print(f'   RMSE  — Train: {rmse_train:>12,.0f}  |  Test: {rmse_test:>12,.0f}')
    print(f'   MAE   — Train: {mae_train:>12,.0f}  |  Test: {mae_test:>12,.0f}')
    print(f'   R²    — Train: {r2_train:>12.4f}  |  Test: {r2_test:>12.4f}')
    print(f'   Overfitting (ΔR²): {overfit:.4f}')

    return {
        'model': name,
        'rmse_train': rmse_train, 'rmse_test': rmse_test,
        'mae_train':  mae_train,  'mae_test':  mae_test,
        'r2_train':   r2_train,   'r2_test':   r2_test,
        'overfitting': overfit
    }

results_all = []

In [ ]:
from sklearn.linear_model import LinearRegression

print('🔧 Entraînement : Linear Regression (baseline)...')
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_train_pred = lr_model.predict(X_train)
lr_test_pred  = lr_model.predict(X_test)

lr_metrics = evaluate_model('Linear Regression', y_test, lr_train_pred, lr_test_pred, y_train)
results_all.append(lr_metrics)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

print('🌲 Entraînement : Random Forest Regressor...')
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred  = rf_model.predict(X_test)

rf_metrics = evaluate_model('Random Forest', y_test, rf_train_pred, rf_test_pred, y_train)
results_all.append(rf_metrics)

In [ ]:
from xgboost import XGBRegressor

print('⚡ Entraînement : XGBoost Regressor (modèle de base)...')
xgb_model = XGBRegressor(random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred  = xgb_model.predict(X_test)

xgb_metrics = evaluate_model('XGBoost (base)', y_test, xgb_train_pred, xgb_test_pred, y_train)
results_all.append(xgb_metrics)

---
## 📈 Étape 6 — Évaluation et comparaison des modèles

Nous comparons les 3 modèles sur les métriques **RMSE**, **MAE** et **R²**, et visualisons :
- Le tableau comparatif
- Les résidus (valeurs réelles vs prédites)
- L'importance des features pour Random Forest et XGBoost

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results_all)
results_df = results_df.sort_values('r2_test', ascending=False)

print('=== Tableau comparatif des modèles (trié par R² Test décroissant) ===')
print(results_df[['model','rmse_test','mae_test','r2_test','overfitting']].round(4).to_string(index=False))

In [ ]:
# Graphique : Valeurs réelles vs prédites pour les 3 modèles
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_preds = [
    ('Linear Regression', lr_test_pred),
    ('Random Forest',     rf_test_pred),
    ('XGBoost (base)',    xgb_test_pred)
]

for ax, (name, preds) in zip(axes, models_preds):
    ax.scatter(y_test, preds, alpha=0.5, s=20, color='steelblue')
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Parfait')
    ax.set_title(f'{name}\nR²={r2_score(y_test, preds):.4f}', fontsize=11)
    ax.set_xlabel('Valeurs réelles')
    ax.set_ylabel('Valeurs prédites')
    ax.legend()

plt.suptitle('Valeurs réelles vs Prédites — comparaison des modèles', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Importance des features : Random Forest vs XGBoost
fi_rf  = pd.Series(rf_model.feature_importances_,  index=FEATURE_COLS).sort_values(ascending=False)
fi_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fi_rf.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Feature Importance — Random Forest', fontsize=12)
axes[0].set_ylabel('Importance')
axes[0].tick_params(axis='x', rotation=45)

fi_xgb.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Feature Importance — XGBoost', fontsize=12)
axes[1].set_ylabel('Importance')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Importance des variables explicatives', fontsize=13)
plt.tight_layout()
plt.show()

---
## 🔧 Étape 7 — Optimisation des hyperparamètres (GridSearchCV)

XGBoost étant le modèle le plus prometteur, nous allons optimiser ses hyperparamètres avec **GridSearchCV** de scikit-learn.

Les hyperparamètres testés :
- `n_estimators` : nombre d'arbres
- `learning_rate` : taux d'apprentissage
- `max_depth` : profondeur maximale des arbres
- `subsample` : fraction des données utilisées par arbre

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

print('⚡ GridSearchCV en cours sur XGBoost... (peut prendre quelques minutes)')

param_grid = {
    'n_estimators':  [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [3, 5, 7],
    'subsample':     [0.8, 1.0]
}

xgb_base = XGBRegressor(random_state=42, verbosity=0)
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\n✅ GridSearchCV terminé !')
print(f'Meilleurs paramètres : {grid_search.best_params_}')
print(f'Meilleur score CV (RMSE) : {-grid_search.best_score_:,.0f}')

In [ ]:
# Évaluation du meilleur modèle optimisé
best_xgb = grid_search.best_estimator_

best_train_pred = best_xgb.predict(X_train)
best_test_pred  = best_xgb.predict(X_test)

best_metrics = evaluate_model(
    'XGBoost (optimisé)', y_test, best_train_pred, best_test_pred, y_train
)
results_all.append(best_metrics)

# Comparaison finale
results_df = pd.DataFrame(results_all).sort_values('r2_test', ascending=False)
print('\n=== Classement final des modèles ===')
print(results_df[['model','rmse_test','mae_test','r2_test','overfitting']].round(4).to_string(index=False))

In [ ]:
# Visualisation des résultats du GridSearch
gs_results = pd.DataFrame(grid_search.cv_results_)
gs_results['rmse'] = -gs_results['mean_test_score']

# Pivot sur learning_rate vs n_estimators (pour max_depth médian)
pivot = gs_results.pivot_table(
    index='param_n_estimators',
    columns='param_learning_rate',
    values='rmse',
    aggfunc='min'
)

plt.figure(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd_r',
            linewidths=0.5, cbar_kws={'label': 'RMSE (CV)'})
plt.title('GridSearchCV — RMSE moyen par combinaison\n(n_estimators vs learning_rate)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 🗄️ Étape 8 — Stockage du meilleur modèle dans le Model Registry

Nous enregistrons les **deux versions** dans le Model Registry Snowflake :
- **V0** : XGBoost de base
- **V1** : XGBoost optimisé (meilleur modèle)

Chaque version est accompagnée de ses métriques et d'une description.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml._internal.utils import identifier

db     = identifier._get_unescaped_name(session.get_current_database())
schema = identifier._get_unescaped_name(session.get_current_schema())

registry   = Registry(session=session, database_name=db, schema_name=schema)
MODEL_NAME = 'HOUSE_PRICE_PREDICTION'

# Données d'exemple pour définir le schéma des features
X_sample = df_train_sp.select(FEATURE_COLS).limit(100)

# --- Version V0 : XGBoost de base ---
print('📦 Enregistrement de V0 (XGBoost base)...')
model_v0 = registry.log_model(
    model_name=MODEL_NAME,
    version_name='V0',
    model=xgb_model,
    sample_input_data=X_sample,
    target_platforms={'WAREHOUSE'}
)
model_v0.set_metric('rmse',  xgb_metrics['rmse_test'])
model_v0.set_metric('mae',   xgb_metrics['mae_test'])
model_v0.set_metric('r2',    xgb_metrics['r2_test'])
model_v0.comment = 'XGBoost Regressor — configuration par défaut (V0 baseline)'

# --- Version V1 : XGBoost optimisé (meilleur modèle) ---
print('📦 Enregistrement de V1 (XGBoost optimisé)...')
model_v1 = registry.log_model(
    model_name=MODEL_NAME,
    version_name='V1',
    model=best_xgb,
    sample_input_data=X_sample,
    target_platforms={'WAREHOUSE'}
)
model_v1.set_metric('rmse',  best_metrics['rmse_test'])
model_v1.set_metric('mae',   best_metrics['mae_test'])
model_v1.set_metric('r2',    best_metrics['r2_test'])
model_v1.comment = (
    f"XGBoost optimisé via GridSearchCV — meilleur modèle. "
    f"Params: {grid_search.best_params_}"
)

print(f'\n✅ Modèle {MODEL_NAME} enregistré avec 2 versions (V0 et V1).')

In [ ]:
# Afficher les versions enregistrées
print('=== Versions du modèle dans le Registry ===')
registry.get_model(MODEL_NAME).show_versions()

In [ ]:
-- Lister tous les modèles dans le registry
SHOW MODELS;

In [ ]:
-- Lister les versions du modèle HOUSE_PRICE_PREDICTION
SHOW VERSIONS IN MODEL HOUSE_PRICE_PREDICTION;

---
## 🎯 Étape 9 — Inférence avec le meilleur modèle

Nous chargeons la version **V1** (meilleure) depuis le registry et l'utilisons pour :
1. Générer des prédictions sur le jeu de test
2. Appeler le modèle via SQL directement
3. Stocker les résultats dans une table Snowflake

In [ ]:
# Chargement de V1 depuis le registry
best_model_ref = registry.get_model(MODEL_NAME).version('V1')

# Inférence sur le jeu de test (Snowpark DataFrame)
test_df_for_inference = df_test_sp.select(FEATURE_COLS)
predictions_sdf = best_model_ref.run(test_df_for_inference, function_name='predict')

print('✅ Inférence (Python API) réussie :')
predictions_sdf.show(10)

In [ ]:
# Sauvegarder les prédictions dans une table Snowflake
# On joint les features + prix réel + prix prédit
predictions_with_actual = df_test_sp.select(FEATURE_COLS + [LABEL_COL]).to_pandas()
predictions_with_actual['PREDICTED_PRICE'] = best_xgb.predict(X_test)
predictions_with_actual['ERROR']           = (
    predictions_with_actual['PREDICTED_PRICE'] - predictions_with_actual[LABEL_COL]
)
predictions_with_actual['ERROR_PCT'] = (
    predictions_with_actual['ERROR'].abs() / predictions_with_actual[LABEL_COL] * 100
).round(2)

# Écriture dans Snowflake
pred_sp = session.create_dataframe(predictions_with_actual)
pred_sp.write.mode('overwrite').save_as_table('HOUSE_PRICE_PREDICTIONS')

print(f'✅ Prédictions sauvegardées dans HOUSE_PRICE_PREDICTIONS ({len(predictions_with_actual)} lignes).')
print(f"Erreur absolue moyenne : {predictions_with_actual['ERROR_PCT'].mean():.2f}%")

In [ ]:
-- Inférence via SQL directement depuis le Model Registry
-- Syntaxe native Snowflake ML
WITH model_alias AS MODEL HOUSE_PRICE_PREDICTION VERSION V1
SELECT
    t.PRICE                                    AS ACTUAL_PRICE,
    model_alias!predict(
        t.AREA, t.BEDROOMS, t.BATHROOMS, t.STORIES,
        t.MAINROAD, t.GUESTROOM, t.BASEMENT, t.HOTWATERHEATING,
        t.AIRCONDITIONING, t.PARKING, t.PREFAREA, t.FURNISHINGSTATUS
    )['output_feature_0']::FLOAT              AS PREDICTED_PRICE,
    ABS(PRICE - model_alias!predict(
        t.AREA, t.BEDROOMS, t.BATHROOMS, t.STORIES,
        t.MAINROAD, t.GUESTROOM, t.BASEMENT, t.HOTWATERHEATING,
        t.AIRCONDITIONING, t.PARKING, t.PREFAREA, t.FURNISHINGSTATUS
    )['output_feature_0']::FLOAT) / PRICE * 100 AS ERROR_PCT
FROM HOUSE_PRICE_TEST t
LIMIT 20;

In [ ]:
-- Aperçu de la table de prédictions stockée
SELECT
    PRICE            AS PRIX_REEL,
    PREDICTED_PRICE  AS PRIX_PREDIT,
    ERROR            AS ECART,
    ERROR_PCT        AS ECART_PCT
FROM HOUSE_PRICE_PREDICTIONS
ORDER BY ERROR_PCT
LIMIT 20;

---
## 🖥️ Étape 10 — Application Streamlit

L'application Streamlit ci-dessous permet aux utilisateurs métier de :
- **Saisir les caractéristiques** d'une maison via des sliders et des listes déroulantes
- **Obtenir une estimation du prix** en temps réel
- **Visualiser** le positionnement du bien par rapport aux données historiques

> ⚠️ Cette cellule doit être **copiée dans une application Streamlit in Snowflake** séparée (menu → Streamlit → New Streamlit App).

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import altair as alt
from snowflake.snowpark.context import get_active_session
import pickle

# ─────────────────────────────────────────
# Configuration de la page
# ─────────────────────────────────────────
st.set_page_config(
    page_title='🏠 House Price Predictor',
    layout='wide'
)

st.title('🏠 House Price Predictor')
st.markdown(
    'Estimez le prix d\'une maison en renseignant ses caractéristiques. '
    'Le modèle **XGBoost optimisé** (V1) depuis le **Snowflake Model Registry** génère la prédiction en temps réel.'
)

# ─────────────────────────────────────────
# Session Snowflake
# ─────────────────────────────────────────
session = get_active_session()

DB_NAME    = 'HOUSE_PRICE_DB'
SCHEMA     = 'ML_SCHEMA'
MODEL_NAME = 'HOUSE_PRICE_PREDICTION'
MODEL_VER  = 'V1'
STAGE_NAME = 'ML_ASSETS'
PIPELINE_FILE = 'preprocessing_pipeline.pkl'

# ─────────────────────────────────────────
# Chargement du pipeline de prétraitement
# ─────────────────────────────────────────
@st.cache_resource
def load_pipeline():
    try:
        session.file.get(f'@{STAGE_NAME}/{PIPELINE_FILE}', '/tmp')
        with open(f'/tmp/{PIPELINE_FILE}', 'rb') as f:
    except Exception as e:
        st.error(f'Impossible de charger le pipeline : {e}')
        return None

@st.cache_data
def load_reference_data():
    return session.table(f'{DB_NAME}.{SCHEMA}.HOUSE_PRICE_RAW').to_pandas()

with st.spinner('Chargement du modèle et des données de référence...'):
    pipeline   = load_pipeline()
    ref_data   = load_reference_data()

if ref_data is None:
    st.stop()

# ─────────────────────────────────────────
# Sidebar — Saisie utilisateur
# ─────────────────────────────────────────
st.sidebar.header('🔧 Caractéristiques de la maison')

st.sidebar.subheader('📐 Caractéristiques physiques')
area       = st.sidebar.slider('Surface (m²)',          int(ref_data['AREA'].min()),     int(ref_data['AREA'].max()),     int(ref_data['AREA'].mean()))
bedrooms   = st.sidebar.slider('Chambres',              int(ref_data['BEDROOMS'].min()), int(ref_data['BEDROOMS'].max()), int(ref_data['BEDROOMS'].median()))
bathrooms  = st.sidebar.slider('Salles de bain',        int(ref_data['BATHROOMS'].min()),int(ref_data['BATHROOMS'].max()),int(ref_data['BATHROOMS'].median()))
stories    = st.sidebar.slider('Étages',                int(ref_data['STORIES'].min()),  int(ref_data['STORIES'].max()),  int(ref_data['STORIES'].median()))
parking    = st.sidebar.slider('Places de parking',     int(ref_data['PARKING'].min()),  int(ref_data['PARKING'].max()),  int(ref_data['PARKING'].median()))

st.sidebar.subheader('🏡 Équipements')
mainroad       = st.sidebar.selectbox('Route principale',       ['yes', 'no'])
guestroom      = st.sidebar.selectbox('Chambre d\'amis',        ['yes', 'no'])
basement       = st.sidebar.selectbox('Sous-sol',               ['yes', 'no'])
hotwaterheating= st.sidebar.selectbox('Chauffe-eau',            ['yes', 'no'])
airconditioning= st.sidebar.selectbox('Climatisation',          ['yes', 'no'])
prefarea       = st.sidebar.selectbox('Zone privilégiée',       ['yes', 'no'])
furnishingstatus = st.sidebar.selectbox('Ameublement',          ['furnished', 'semi-furnished', 'unfurnished'])

# ─────────────────────────────────────────
# Encodage des inputs
# ─────────────────────────────────────────
def encode_input(area, bedrooms, bathrooms, stories, mainroad, guestroom,
                 basement, hotwaterheating, airconditioning, parking,
                 prefarea, furnishingstatus):
    binary_map = {'yes': 1, 'no': 0}
    furnish_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
    # Normalisation de l'AREA (même logique que le pipeline : MinMax)
    area_min = float(ref_data['AREA'].min())
    area_max = float(ref_data['AREA'].max())
    area_norm = (area - area_min) / (area_max - area_min)
    return pd.DataFrame([{
        'AREA':             area_norm,
        'BEDROOMS':         bedrooms,
        'BATHROOMS':        bathrooms,
        'STORIES':          stories,
        'MAINROAD':         binary_map[mainroad],
        'GUESTROOM':        binary_map[guestroom],
        'BASEMENT':         binary_map[basement],
        'HOTWATERHEATING':  binary_map[hotwaterheating],
        'AIRCONDITIONING':  binary_map[airconditioning],
        'PARKING':          parking,
        'PREFAREA':         binary_map[prefarea],
        'FURNISHINGSTATUS': furnish_map[furnishingstatus],
    }])

# ─────────────────────────────────────────
# Prédiction
# ─────────────────────────────────────────
if st.sidebar.button('🔮 Estimer le prix', type='primary'):
    with st.spinner('Génération de la prédiction...'):
        try:
            # Encodage des features
            input_encoded = encode_input(
                area, bedrooms, bathrooms, stories,
                mainroad, guestroom, basement, hotwaterheating,
                airconditioning, parking, prefarea, furnishingstatus
            )
            values_str = ', '.join(map(str, input_encoded.iloc[0].tolist()))

            # Appel SQL au modèle dans le registry
            sql = f"""
                WITH mv AS MODEL {MODEL_NAME} VERSION {MODEL_VER}
                SELECT mv!predict(
                    t.AREA, t.BEDROOMS, t.BATHROOMS, t.STORIES,
                    t.MAINROAD, t.GUESTROOM, t.BASEMENT, t.HOTWATERHEATING,
                    t.AIRCONDITIONING, t.PARKING, t.PREFAREA, t.FURNISHINGSTATUS
                )['output_feature_0']::FLOAT AS PREDICTED_PRICE
                FROM (VALUES ({values_str})) AS t(
                    AREA, BEDROOMS, BATHROOMS, STORIES,
                    MAINROAD, GUESTROOM, BASEMENT, HOTWATERHEATING,
                    AIRCONDITIONING, PARKING, PREFAREA, FURNISHINGSTATUS
                )
            """
            result = session.sql(sql).collect()
            predicted_price = float(result[0]['PREDICTED_PRICE'])

            # ── Affichage du résultat ──
            st.success('✅ Estimation générée avec succès !')

            col1, col2, col3 = st.columns(3)
            col1.metric('💰 Prix estimé',           f'{predicted_price:,.0f} €')
            col2.metric('📊 Prix médian du marché', f"{ref_data['PRICE'].median():,.0f} €")
            col3.metric(
                '📈 Écart vs médian',
                f"{((predicted_price - ref_data['PRICE'].median()) / ref_data['PRICE'].median() * 100):+.1f}%"
            )

            # ── Positionnement dans la distribution ──
            st.subheader('📍 Positionnement dans la distribution des prix')
            price_data = pd.DataFrame({'PRICE': ref_data['PRICE']})
            hist_chart = alt.Chart(price_data).mark_bar(opacity=0.6, color='steelblue').encode(
                alt.X('PRICE:Q', bin=alt.Bin(maxbins=40), title='Prix (€)'),
                alt.Y('count()', title='Nombre de maisons')
            )
            pred_line = alt.Chart(pd.DataFrame({'x': [predicted_price]})).mark_rule(
                color='red', strokeWidth=2.5, strokeDash=[6, 3]
            ).encode(x='x:Q')
            st.altair_chart((hist_chart + pred_line).properties(
                height=300, title='Distribution des prix — ligne rouge = votre estimation'
            ), use_container_width=True)

            # ── Résumé des caractéristiques saisies ──
            st.subheader('📋 Récapitulatif des caractéristiques saisies')
            summary = {
                'Caractéristique': [
                    'Surface (m²)', 'Chambres', 'Salles de bain', 'Étages', 'Parking',
                    'Route principale', 'Chambre d\'amis', 'Sous-sol', 'Chauffe-eau',
                    'Climatisation', 'Zone privilégiée', 'Ameublement'
                ],
                'Valeur': [
                    area, bedrooms, bathrooms, stories, parking,
                    mainroad, guestroom, basement, hotwaterheating,
                    airconditioning, prefarea, furnishingstatus
                ]
            }
            st.dataframe(pd.DataFrame(summary), use_container_width=True)

        except Exception as e:
            st.error(f'❌ Erreur lors de la prédiction : {e}')
else:
    st.info('👈 Renseignez les caractéristiques de la maison dans le panneau latéral, puis cliquez sur **Estimer le prix**.')

    # Aperçu des données historiques
    st.subheader('📊 Aperçu des données historiques')
    col1, col2, col3, col4 = st.columns(4)
    col1.metric('Nombre de biens',   len(ref_data))
    col2.metric('Prix moyen',        f"{ref_data['PRICE'].mean():,.0f} €")
    col3.metric('Prix min',          f"{ref_data['PRICE'].min():,.0f} €")
    col4.metric('Prix max',          f"{ref_data['PRICE'].max():,.0f} €")

    price_dist = alt.Chart(ref_data).mark_bar(color='steelblue').encode(
        alt.X('PRICE:Q', bin=alt.Bin(maxbins=40), title='Prix (€)'),
        alt.Y('count()', title='Nombre de maisons')
    ).properties(height=300, title='Distribution des prix de vente')
    st.altair_chart(price_dist, use_container_width=True)

---
## ✅ Conclusion

### Récapitulatif du pipeline MLOps réalisé

| Étape | Action | Outil Snowflake |
|-------|--------|-----------------|
| 1 | Configuration de l'environnement | SQL DDL — Database, Schema, Stage |
| 2 | Ingestion depuis S3 | `session.read.csv()` + Snowpark |
| 3 | Exploration (EDA) | Snowpark + Pandas + Matplotlib/Seaborn |
| 4 | Feature Engineering | `snowflake.ml.modeling.preprocessing` + Pipeline |
| 5 | Entraînement (3 modèles) | scikit-learn + XGBoost |
| 6 | Évaluation (RMSE, MAE, R²) | sklearn.metrics |
| 7 | Optimisation (GridSearchCV) | sklearn + XGBoost |
| 8 | Model Registry | `snowflake.ml.registry.Registry` |
| 9 | Inférence SQL + Python | Registry API + SQL natif Snowflake |
| 10 | Application Streamlit | Streamlit in Snowflake |

### Résultats obtenus
Le modèle **XGBoost optimisé (V1)** enregistré dans le **Snowflake Model Registry** constitue le meilleur modèle. Les prédictions sont stockées dans la table `HOUSE_PRICE_PREDICTIONS` et accessibles via l'application Streamlit.

---
*Livrable — MBAESG — Évaluation Data Engineering & MLOps — Snowflake*